# 파일럿 — AlphaMix+(PRUDEX-Compass, TMLR 2023) 공식 코드 실행 가능성

**목적(사전 고정).** 재현·재평가 대상(T2)으로 쓸 수 있는지 **실행 가능성만** 확인한다: (1) 설치·임포트 (2) US(dj30) 데이터 구성 (3) 시드 1개(12345) 학습 시간 (4) 테스트 TR을 논문 Table 5 US 2021(시드 5개 평균 TR 20.7%, SR 1.64)과 나란히 기록 — **판정 아님**.

**하지 않는 것.** 비용 적용, 등가중 비교, 시작 상태 변경, 다른 시드·단계.

**대상 코드.** `TradeMaster-NTU/PRUDEX-Compass` @ `2b62831`, `AlphaMix+/PM/`. 저장소 스크립트 `AlphaMix+_sz50.py`의 `experiment()`를 그대로 쓰고, 실행 설정(variant)만 스크립트에 **주석으로 남아 있는 dj_30 값**(num_trains_per_train_loop=2000, num_expl_steps_per_train_loop=2000, min_num_steps_before_training=70000)으로 바꾼다. 나머지(10 epoch, 앙상블 3, 학습률 등)는 스크립트 그대로.

**호환 shim(동작을 바꾸지 않는 최소 수정, `sitecustomize.py`로 주입).**
1. `pandas.DataFrame.append`(pandas 2에서 제거) → `pd.concat`로 같은 의미 구현.
2. `yfinance.download` → 2023년 형식(단일 열 인덱스, `Open, High, Low, Close, Adj Close, Volume` 순서, 수정주가 열 포함)으로 반환하도록 `auto_adjust=False`, 열 순서 고정. **데이터는 오늘 받은 Yahoo 값이라 저자가 받은 값과 다를 수 있다(배당·분할 재조정).**
3. `gym.Env.seed`(gym 0.26에서 제거) → 환경의 `_seed`를 호출(구버전 동작).
4. GPU가 없으면 `ptu.set_gpu_mode(True)` 대신 CPU 사용.

**런타임.** GPU(T4) 권장 — 원 코드가 GPU를 가정한다. 결과는 Drive `data/alphamix_pilot/`. 끊기면 "모두 실행"으로 다시 돌린다(완료된 단계는 건너뜀).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = pathlib.Path('/content/drive/MyDrive/data/alphamix_pilot'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
COMMIT = '2b62831'
!test -d /content/PRUDEX || git clone -q https://github.com/TradeMaster-NTU/PRUDEX-Compass.git /content/PRUDEX
!cd /content/PRUDEX && git checkout -q $COMMIT && git log -1 --format='%H %cd'
!pip install -q gym==0.26.2 gtimer exchange_calendars stockstats yfinance python-dateutil cloudpickle

In [ ]:
# 호환 shim — 위 1~4. 하위 프로세스에서도 적용되도록 sitecustomize로 넣는다
SHIM = r'''
import pandas as pd
if not hasattr(pd.DataFrame, "append"):
    def _append(self, other, ignore_index=False, **kw):
        return pd.concat([self, other], ignore_index=ignore_index)
    pd.DataFrame.append = _append
try:
    import yfinance as yf
    _dl = yf.download
    def _download(tic, *a, **kw):
        kw.setdefault("auto_adjust", False); kw.setdefault("progress", False)
        df = _dl(tic, *a, **kw)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
    yf.download = _download
except ImportError:
    pass
try:
    import gym
    if not hasattr(gym.Env, "seed"):
        gym.Env.seed = lambda self, seed=None: self._seed(seed) if hasattr(self, "_seed") else [seed]
except ImportError:
    pass
'''
SITE = pathlib.Path('/content/shim'); SITE.mkdir(exist_ok=True); (SITE / 'sitecustomize.py').write_text(SHIM)
(OUT / 'shim.py').write_text(SHIM)

RUN = r'''
import sys, json, time, importlib.util, torch
sys.path.insert(0, "/content/PRUDEX/AlphaMix+/PM")
spec = importlib.util.spec_from_file_location("am", "/content/PRUDEX/AlphaMix+/PM/AlphaMix+_sz50.py")
am = importlib.util.module_from_spec(spec); spec.loader.exec_module(am)   # __main__ 블록은 실행되지 않음
import rlkit.torch.pytorch_util as ptu
from rlkit.launchers.launcher_util import setup_logger_custom
mode, seed = sys.argv[1], int(sys.argv[2])
if mode == "data":
    from rl_env.portfolio_management import get_dataset_config
    tr, va, te = get_dataset_config("dj30")
    print(json.dumps({k: [str(c["df"].date.min()), str(c["df"].date.max()), int(c["stock_dim"]), float(c["transaction_cost_pct"])]
                      for k, c in zip(["train", "valid", "test"], [tr, va, te])}))
    sys.exit()
variant = dict(algorithm="SAC", version="normal", layer_size=128, buffer_size=int(1E4),
    algorithm_kwargs=dict(num_epochs=10, num_eval_steps_per_epoch=1500,
        num_trains_per_train_loop=2000, num_expl_steps_per_train_loop=2000, min_num_steps_before_training=70000,  # 스크립트 주석의 dj_30 값
        max_path_length=5000, batch_size=256, save_frequency=0),
    trainer_kwargs=dict(discount=0.99, soft_target_tau=5e-3, target_update_period=1, policy_lr=7E-4, qf_lr=7E-4,
        reward_scale=1, use_automatic_entropy_tuning=True),
    num_ensemble=3, num_layer=2, dataset="dj30", seed=seed, ber_mean=0.5, env="portfolio_management",
    inference_type=0.0, temperature=20.0, log_dir="", uncertainty=0.5)
gpu = torch.cuda.is_available(); ptu.set_gpu_mode(gpu)
log_dir = setup_logger_custom("AlphaMix+", variant=variant); variant["log_dir"] = log_dir
ptu.set_gpu_mode(gpu)
t0 = time.time(); am.experiment(variant)
print("__LOGDIR__", log_dir); print("__ELAPSED__", round(time.time() - t0, 1)); print("__GPU__", gpu)
'''
(pathlib.Path('/content/run_am.py')).write_text(RUN)
import os, subprocess, sys
ENVV = dict(os.environ, PYTHONPATH='/content/shim:/content/PRUDEX/AlphaMix+/PM')
def sh(args, tag):
    p = subprocess.run([sys.executable, '/content/run_am.py'] + args, capture_output=True, text=True, env=ENVV,
                       cwd='/content/PRUDEX/AlphaMix+/PM')
    (OUT / f'{tag}.log').write_text(p.stdout + '\n--- stderr ---\n' + p.stderr)
    print(p.stdout[-2500:]); print('stderr 끝부분:', p.stderr[-2500:]); return p
import torch, platform
print(dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.cuda.is_available()))

## A. 데이터 구성(dj30) — 임포트·다운로드·분할 확인

In [ ]:
%%time
pa = sh(['data', '0'], 'A_data')

## B. 시드 12345 학습 1회(10 epoch) — 소요 시간과 epoch별 테스트 TR

In [ ]:
%%time
import pandas as pd, re, shutil
done = OUT / 'B_result.csv'
if done.exists():
    print('이미 완료 — 건너뜀'); print(pd.read_csv(done))
else:
    pb = sh(['train', '12345'], 'B_train_seed12345')
    ld = re.search(r'__LOGDIR__ (.+)', pb.stdout).group(1).strip()
    shutil.copy(f'{ld}/result.csv', done)
    for f in pathlib.Path(ld).glob('test_daily_return_*.csv'): shutil.copy(f, OUT / f.name)
    print(pd.read_csv(done))

In [ ]:
# 요약 — 논문 Table 5 US 2021 AlphaMix+: TR 20.7%, SR 1.64 (시드 5개 평균; 판정 아님)
import re
txt = (OUT / 'B_train_seed12345.log').read_text()
el = float(re.search(r'__ELAPSED__ ([\d.]+)', txt).group(1)); gpu = re.search(r'__GPU__ (\w+)', txt).group(1)
r = pd.read_csv(OUT / 'B_result.csv')
print(f'GPU={gpu}, 학습 1회 {el/60:.1f}분 → 시드 5개 × 3단계 약 {15*el/3600:.1f}시간')
print('epoch 10 test TR(비율), SR:', r['test_tr'].iloc[-1], r['test_sr'].iloc[-1])
print(r)
print('\n이 셀 출력을 Claude에게 붙여 넣거나 Drive 폴더를 알려 주세요.')